In [1]:
import pandas as pd
import duckdb

# Make monthly sales from sales.csv

## Load sales.csv 

- Test data generated by script Grok made.
- 2 suppliers with 2 licensees each, plus another licensee, who buys from both suppliers.
- Date range 3/2023 to 1/2026

In [2]:
sales = pd.read_csv("./csv/sales-grok.csv", parse_dates=['DATE'])
sales['AMT'] = sales['AMT'].astype('int64')
sales['QTY'] = sales['QTY'].astype('int64')
sales

,DATE,SUPPLIER,LICENSEE,PO,PART,QTY,AMT
0,2023-03-08,nippon-metal,ninja-roofing,PO0001,roof-polish,57,11100
1,2023-03-08,nippon-metal,ninja-roofing,PO0001,glass-decoration,114,86100
2,2023-03-08,nippon-metal,ninja-roofing,PO0001,gold-clip,156,89500
3,2023-03-08,nippon-metal,ninja-roofing,PO0001,glass-clip,109,128600
4,2023-03-08,nippon-metal,ninja-roofing,PO0001,gold-decoration,165,64400
...,...,...,...,...,...,...,...
872,2026-01-04,nippon-metal,global-roof,PO0168,tin-clip,55,5800
873,2026-01-15,us-steel,ranger-roofing,PO0169,glass-clip,142,178100
874,2026-01-15,us-steel,ranger-roofing,PO0169,tin-clip,219,24600
875,2026-01-15,us-steel,ranger-roofing,PO0169,roof-polish,117,27800


## Group sales by month, supplier, licensee, and part.  
## Calculate monthly and YTD totals.

In [3]:
duckdb.query("DROP VIEW IF EXISTS monthly_sales");

duckdb.query("""
CREATE VIEW monthly_sales AS
SELECT 
    strftime('%Y-%m', date) AS month,
    supplier,
    licensee,
    part,
    CAST(SUM(qty) AS INT64) AS qty,
    CAST(SUM(amt) AS INT64) AS month_amt,
    CAST(
        SUM(month_amt) OVER (
            PARTITION BY SUBSTR(month, 1, 4), supplier, licensee, part 
            ORDER BY month
        ) AS INT64
    ) AS ytd_amt
FROM sales
GROUP BY month, supplier, licensee, part
ORDER BY month, supplier, licensee, part;
""")

ytd_df = duckdb.query("select * from monthly_sales").df()

ytd_df

,month,SUPPLIER,LICENSEE,PART,qty,month_amt,ytd_amt
0,2023-03,nippon-metal,ninja-roofing,glass-clip,109,128600,128600
1,2023-03,nippon-metal,ninja-roofing,glass-decoration,114,86100,86100
2,2023-03,nippon-metal,ninja-roofing,gold-clip,156,89500,89500
3,2023-03,nippon-metal,ninja-roofing,gold-decoration,165,64400,64400
4,2023-03,nippon-metal,ninja-roofing,roof-polish,57,11100,11100
...,...,...,...,...,...,...,...
872,2026-01,nippon-metal,rice-roofers,tin-decoration,144,16300,16300
873,2026-01,us-steel,ranger-roofing,glass-clip,142,178100,178100
874,2026-01,us-steel,ranger-roofing,gold-clip,120,75800,75800
875,2026-01,us-steel,ranger-roofing,roof-polish,117,27800,27800


## Save to csv.

In [4]:
ytd_df.to_csv('./csv/monthly-sales.csv', index=False)